# Agentic RAG — Corrective, Adaptive & ReAct (4 of 6)

## RAG Workshop Series

This notebook is part of a 6-notebook series (split from the original `RAG.ipynb`), each one runnable on its own in Google Colab:

1. **`basic_RAG.ipynb`** — naive vector RAG: chunk → embed → cosine similarity → prompt → LLM
2. **`hybrid_search_RAG.ipynb`** — BM25 keyword search + semantic search fusion + cross-encoder reranking
3. **`query_and_chunking_RAG.ipynb`** — query rewriting, advanced chunking strategies, metadata filtering
4. **`agentic_RAG.ipynb`** — Corrective RAG (CRAG), Adaptive RAG (routing), Agentic RAG (ReAct loop)
5. **`pdf_chroma_RAG.ipynb`** — build RAG over a real PDF, store vectors persistently in Chroma
6. **`rag_when_to_use.ipynb`** — reference: when RAG is (and isn't) the right tool

Each notebook installs its own dependencies and rebuilds whatever context it needs, so you can open any one directly without running the others first.

## What's in this notebook

These three techniques make retrieval a **loop with decisions** instead of a single fixed step:

- **Corrective RAG (CRAG)** — evaluate whether retrieved chunks are actually relevant; rewrite the query and retry if not
- **Adaptive RAG** — route each question to RAG or straight to the LLM, depending on whether it's in-domain
- **Agentic RAG (ReAct)** — let the LLM decide, step by step, whether to search again or answer

All three reuse hybrid search + reranking from `hybrid_search_RAG.ipynb`, so this notebook rebuilds that pipeline first.

## Setup

Needs an `NSCALE_API` key as a Colab secret (🔑 icon in the left sidebar) — same as `query_and_chunking_RAG.ipynb`.

In [ ]:
!pip install -q sentence-transformers rank_bm25 openai

In [ ]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(
    api_key=userdata.get('NSCALE_API'),
    base_url="https://inference.api.nscale.com/v1",
)

### Rebuild the hybrid search + reranking pipeline

In [ ]:
%%capture
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

In [ ]:
def chunk_text(text, chunk_size=250):
    words = text.split()

    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks

In [ ]:
from rank_bm25 import BM25Okapi

def process_documents(documents, embedding_model, chunk_size=250):
    # Chunk the documents
    all_chunks = []
    for document in documents:
        all_chunks.extend(chunk_text(document, chunk_size))

    # Generate chunk embeddings
    chunk_embeddings = embedding_model.encode(all_chunks)

    # Tokenize chunks for BM25
    tokenized_chunks = [
        chunk.lower().split()
        for chunk in all_chunks
    ]
    bm25_index = BM25Okapi(tokenized_chunks)

    return all_chunks, chunk_embeddings, bm25_index

In [ ]:
def hybrid_search(question, top_k=2, semantic_weight=0.5, keyword_weight=0.5):

    # -------------------------
    # 1. Semantic Search
    # -------------------------

    question_embedding = embedding_model.encode([question])

    semantic_scores = cosine_similarity(
        question_embedding,
        chunk_embeddings
    )[0]

    # -------------------------
    # 2. Keyword Search (BM25)
    # -------------------------

    query_tokens = question.lower().split()

    bm25_scores = bm25.get_scores(query_tokens)

    # -------------------------
    # 3. Normalize Scores
    # -------------------------

    semantic_normalized = (
        semantic_scores - semantic_scores.min()
    ) / (
        semantic_scores.max() - semantic_scores.min()
    )

    bm25_normalized = (
        bm25_scores - bm25_scores.min()
    ) / (
        bm25_scores.max() - bm25_scores.min()
    )

    # -------------------------
    # 4. Combine Scores
    # -------------------------

    hybrid_scores = (
        semantic_weight * semantic_normalized
        + keyword_weight * bm25_normalized
    )

    # -------------------------
    # 5. Get Top Results
    # -------------------------

    top_indices = hybrid_scores.argsort()[::-1][:top_k]

    results = []

    for index in top_indices:
        results.append({
            "text": chunks[index],
            "semantic_score": float(semantic_normalized[index]),
            "bm25_score": float(bm25_normalized[index]),
            "hybrid_score": float(hybrid_scores[index])
        })

    return results

In [ ]:
def rerank(question, results, top_k=2):

    # Create query-document pairs
    pairs = [
        (question, result["text"])
        for result in results
    ]

    # Calculate relevance scores
    scores = reranker.predict(pairs)

    # Add scores to results
    for result, score in zip(results, scores):
        result["reranker_score"] = float(score)

    # Sort by reranker score
    results = sorted(
        results,
        key=lambda x: x["reranker_score"],
        reverse=True
    )

    return results[:top_k]

### Knowledge base (extended AWS services corpus)

In [ ]:
documents = [
    """
    Amazon SQS is a managed message queue service.

    It allows applications to communicate asynchronously.
    Producers send messages to a queue and consumers process
    those messages independently.

    SQS is commonly used to decouple distributed applications
    so that one service does not have to wait for another service
    to finish processing.
    """,

    """
    AWS Lambda is a serverless compute service.

    Developers upload code and AWS executes the code in response
    to events. Lambda automatically manages servers and scales
    applications based on incoming requests.
    """,

    """
    Amazon S3 is an object storage service designed for storing
    and retrieving files.

    It is commonly used for backups, static websites, data lakes,
    application assets, and large collections of unstructured data.
    """,

    """
    Amazon DynamoDB is a managed NoSQL database.

    It provides low-latency access to data and automatically
    scales to handle large workloads.

    DynamoDB is commonly used for applications that require
    fast access to large amounts of structured data.
    """,

    """
    Amazon API Gateway is a managed service for creating,
    publishing, monitoring, and securing APIs.

    It can route incoming HTTP requests to backend services
    such as AWS Lambda.
    """,

    """
    Amazon SNS is a publish-subscribe messaging service.

    It allows applications to send notifications to multiple
    subscribers or downstream systems.

    SNS is commonly used for event broadcasting and notifications.
    """,

    """
    AWS CloudWatch provides monitoring and observability for
    AWS resources and applications.

    It collects metrics, logs, and events and can trigger
    alarms when specified conditions occur.
    """,

    """
    AWS IAM manages access to AWS resources.

    It allows administrators to define users, roles, policies,
    and permissions controlling which resources applications
    and users can access.
    """
]
# Now, call the new function to process the current documents
chunks, chunk_embeddings, bm25 = process_documents(documents, embedding_model)

## Step 14: Corrective RAG (CRAG)

In [ ]:
def evaluate_retrieval(question, results):

    context = "\n\n".join(
        result["text"]
        for result in results
    )

    prompt = f"""
You are evaluating whether retrieved documents are useful for answering
a user's question. It could be only one sentence that's relevant, it is still relevant.

Question:
{question}

Retrieved documents:
{context}

Return ONLY one of these:
RELEVANT
NOT_RELEVANT

Are the retrieved documents relevant enough to answer the question?
"""

    response = client.chat.completions.create(
        model="meta-llama/Llama-3.1-8B-Instruct",
        messages=[
            {
                "role": "system",
                "content": "You evaluate retrieval quality."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_completion_tokens=50,
        reasoning_effort="low",
        temperature=0
    )

    return response.choices[0].message.content.strip()

`corrective_rag()` below rewrites the query and retries when retrieval quality is poor. That reuses `rewrite_query()` from `query_and_chunking_RAG.ipynb` (Step 11) — redefined here so this notebook doesn't depend on that one.

In [ ]:
def rewrite_query(question):

    response = client.chat.completions.create(
        model="meta-llama/Llama-3.1-8B-Instruct",
        messages=[
            {
                "role": "system",
                "content": """
Rewrite the user's question into a concise search query with given context.
Return only the rewritten query.
Do not answer the question.
"""
            },
            {
                "role": "user",
                "content": question
            }
        ],
        max_tokens=100,
        reasoning_effort="low",
        temperature=0.2,
    )

    return response.choices[0].message.content or ""

In [ ]:
def corrective_rag(question):

    # 1. Retrieve
    results = hybrid_search(
        question,
        top_k=5
    )

    # 2. Rerank
    results = rerank(
        question,
        results,
        top_k=3
    )

    # 3. Evaluate retrieval
    evaluation = evaluate_retrieval(
        question,
        results
    )

    print("Retrieval:", evaluation)

    # 4. Correct if retrieval is poor
    if "NOT_RELEVANT" in evaluation:

        print("Correcting retrieval...")

        rewritten_query = rewrite_query(question)

        results = hybrid_search(
            rewritten_query,
            top_k=5
        )

        results = rerank(
            rewritten_query,
            results,
            top_k=3
        )

    # 5. Return retrieved context
    return results

Try it on a question the knowledge base can actually answer:

In [ ]:
results = corrective_rag(
    "How can I make two services work independently so one doesn't have to wait for the other?"
)

for result in results:
    print(result["text"])
    print()

Now try it on an out-of-domain question:

In [ ]:
results = corrective_rag(
    "Why is sky blue?"
)

for result in results:
    print(result["text"])
    print()

# Adaptive RAG
### Routing

Question ---> Router --->RAG or Direct

In [ ]:
def route_question(question):

    knowledge_base = """
    Our knowledge base contains information about AWS services,
    including Amazon S3, Amazon SQS, AWS Lambda, Amazon DynamoDB,
    Amazon API Gateway, Amazon SNS, and AWS IAM.
    """

    prompt = f"""
You are a query router.

Decide whether the user's question should be answered using
the knowledge base or directly by the LLM.

Knowledge base:
{knowledge_base}

Return ONLY:
RAG
or
DIRECT

Use RAG if the question is about information covered by the
knowledge base.

Use DIRECT if the question is unrelated to the knowledge base.

Question:
{question}
"""

    response = client.chat.completions.create(
        model="meta-llama/Llama-3.1-8B-Instruct",
        messages=[
            {
                "role": "system",
                "content": "You are a query routing classifier."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=20,
        temperature=0
    )

    return response.choices[0].message.content.strip()

In [ ]:
def adaptive_rag(question):

    route = route_question(question)

    print("Route:", route)

    if "DIRECT" in route:
        prompt = question

    else:
        results = hybrid_search(
            question,
            top_k=5
        )

        results = rerank(
            question,
            results,
            top_k=3
        )

        context = "\n\n".join(
            result["text"]
            for result in results
        )

        prompt = f"""
Answer the question using the provided context.

Context:
{context}

Question:
{question}
"""

    response = client.chat.completions.create(
        model="meta-llama/Llama-3.1-8B-Instruct",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_completion_tokens=300,
        reasoning_effort="low",
        temperature=0
    )

    return response.choices[0].message

In [ ]:
questions = [
    "What is AWS SQS?",
    "What is the capital of France?",
    "How does Lambda handle incoming requests?",
    "What is 25 multiplied by 4?"
]

for question in questions:

    print("\nQuestion:", question)

    answer = adaptive_rag(question)

    print("Answer:", answer)

## Agentic RAG — ReAct (Reasoning + Act)

Question → Agent → Search → Evaluate → Search Again? → Evaluate → Answer

First pass: a simple ReAct loop.

In [ ]:
def agentic_rag(question, max_steps=3):

    conversation = [
        {
            "role": "system",
            "content": """
You are a RAG agent.

You have access to a knowledge base through the SEARCH action.

For each step, decide what to do.

Return exactly one of:

SEARCH: <search query>
ANSWER: <final answer>

Use SEARCH when you need information from the knowledge base.
Use ANSWER when you have enough information to answer the question.
"""
        },
        {
            "role": "user",
            "content": question
        }
    ]

    for step in range(max_steps):

        response = client.chat.completions.create(
            model="meta-llama/Llama-3.1-8B-Instruct",
            messages=conversation,
            max_tokens=200,
            temperature=0
        )

        decision = response.choices[0].message.content.strip()

        print(f"\nStep {step + 1}:")
        print(decision)

        # Agent wants to search
        if decision.startswith("SEARCH:"):

            search_query = decision.replace(
                "SEARCH:", ""
            ).strip()

            results = hybrid_search(
                search_query,
                top_k=5
            )

            results = rerank(
                search_query,
                results,
                top_k=3
            )

            context = "\n\n".join(
                result["text"]
                for result in results
            )

            conversation.append({
                "role": "assistant",
                "content": decision
            })

            conversation.append({
                "role": "user",
                "content": f"""
Search results:

{context}

Decide whether you have enough information.
If not, search again.
"""
            })

        # Agent has enough information
        elif decision.startswith("ANSWER:"):

            return decision.replace(
                "ANSWER:",
                ""
            ).strip()

    return "I could not find enough information."

In [ ]:
question = """
How can AWS services communicate asynchronously
without requiring one service to wait for another?
"""

answer = agentic_rag(question)

print("\nFinal Answer:")
print(answer)

The loop above can search the same query forever if the agent gets stuck. Here's an improved version that tracks previous queries and refuses to repeat one:

In [ ]:
def agentic_rag(question, max_steps=3, llm_model_name="meta-llama/Llama-3.1-8B-Instruct"):

    messages = [
        {
    "role": "system",
    "content": """
You are a RAG agent.

You can search a knowledge base.

IMPORTANT:
You may ONLY use facts explicitly stated in the search results.
Do not use your own knowledge.
Do not add facts that are not present in the search results.

After seeing search results:

If the results contain enough information:
ANSWER: <answer>

If information is missing:
SEARCH: <new search query>

Do not search again if the answer can be supported by the results.
Do not repeat a previous search query.
"""
},
        {
            "role": "user",
            "content": question
        }
    ]

    previous_queries = []

    for step in range(max_steps):

        response = client.chat.completions.create(
            model=llm_model_name,
            messages=messages,
            max_tokens=200,
            temperature=0
        )

        decision = response.choices[0].message.content.strip()
        print(decision)

        print(f"Step {step + 1}: {decision}")

        # -------------------------
        # Agent wants to answer
        # -------------------------
        if decision.startswith("ANSWER:"):

            return decision.replace(
                "ANSWER:",
                ""
            ).strip()

        # -------------------------
        # Agent wants to search
        # -------------------------
        if decision.startswith("SEARCH:"):

            search_query = decision.replace(
                "SEARCH:",
                ""
            ).strip()

            # Prevent repeated searches
            if search_query.lower() in [
                q.lower() for q in previous_queries
            ]:
                return "The agent could not find enough information."

            previous_queries.append(search_query)

            results = hybrid_search(
                search_query,
                top_k=5
            )

            results = rerank(
                search_query,
                results,
                top_k=3
            )

            context = "\n\n".join(
                result["text"]
                for result in results
            )

            messages.append({
                "role": "assistant",
                "content": decision
            })

            messages.append({
                "role": "user",
                "content": f"""
Search results for:

{search_query}

{context}

Now decide:

ANSWER: <answer>

or

SEARCH: <different search query>
"""
            })

    return "The agent could not find enough information."

In [ ]:
llm_model_name="meta-llama/Llama-3.1-8B-Instruct"

question = """
How can AWS services communicate asynchronously without requiring one service to wait for another?
"""
answer = agentic_rag(question, llm_model_name=llm_model_name)

print("\nFinal Answer:")
print(answer)

---

➡️ **Next:** [`pdf_chroma_RAG.ipynb`](./pdf_chroma_RAG.ipynb) — build RAG over a real PDF and persist the vectors in Chroma instead of a Python list.